In [ ]:
# We rely on several libraries
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import sklearn
import time

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

In [ ]:
# Magic formulae so that changes in the .py files are loaded in here
%load_ext autoreload
%autoreload 2
###

    
# We load in some code we've defined elsewhere 
from src.weather_helpers import (
    WeatherDataset, filepath, metapath, reset_seeds, plot_learning_curves, get_class_name
)

from src.define_model import Net

from src.train_and_evaluate import evaluate_model, evaluate_wrapper, train_model

## Problem introduction

This project tries to classify images based on the weather shown. For instance, the below image should be classified as "snow". 

The code we use for this project is based on the CNN example in the PyTorch documentation, 
https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html

Our dataset comes from Kaggle: https://www.kaggle.com/datasets/jehanbhathena/weather-dataset

In [ ]:
Image.open(filepath+'snow/'+'0840.jpg')


In [ ]:
Image.open(filepath+'rainbow/'+'0592.jpg')


In [ ]:
Image.open(filepath+'lightning/'+'1830.jpg')


### Data handling

This data set is bigger than the earlier ones: we have several thousand images, and we don't want to keep all of them in memory, or show them all to the model at the same time. Therefore, we use the below data set and data loader wrappers to our data.

In [ ]:
# We have some objects that hold our data.
# The data split has already been done for us 
start = time.time()
D = 32
trainset = WeatherDataset(filepath, metapath, subset='train', D=D)
valset = WeatherDataset(filepath, metapath, subset='val', D=D)
print(f'Data loading took {np.round(time.time()-start, 4)} seconds.')

In [ ]:
B = 64
# Pytorch provides data loaders, which helpfully groups our data into batches
# Here we also choose to shuffle the training data, but not the validation data
train_dataloader = DataLoader(trainset, batch_size=B, shuffle=True)
val_dataloader = DataLoader(valset, batch_size=B, shuffle=False)

In [ ]:
# Debugging: Make sure we have a GPU available 
print(torch.cuda.is_available())
device = torch.device('cuda')

### Examples images, as represented by the model
How much detail these images have is controlled by the hyperparameter **D** that's passed to `WeatherDataset`. You can play with this! What happens if we set `D=64`? Does the training time change? Does the accuracy change?

In [ ]:
print(get_class_name(trainset._labels[10]))
plt.imshow(np.array(trainset._loaded_images[10].T, dtype=int));

In [ ]:
print(get_class_name(trainset._labels[500]))
plt.imshow(np.array(trainset._loaded_images[500].T, dtype=int));

## Baseline

In [ ]:
# Let's come up with a baseline:
# What accuracy would we get by randomly guessing the class for each image?
reset_seeds(1) # Make things reproducible
random_preds = np.random.randint(0, 11, len(valset))
# Make an array with all the validation true labels
val_labels = np.hstack([ll.numpy() for _, ll in val_dataloader])

# Compare the true and random labels
rand_acc = 100*np.sum(random_preds == val_labels)/len(val_labels)
print(f'Random accuracy: {rand_acc:.4f}%')

## Let's try some models!


### Model 1

In [ ]:
epochs =  3



In [ ]:
reset_seeds(1) # Make things reproducible
net = Net(D=D) # Set up our model
net.to(device) # Send it to the gpu
# Let's choose some hyperparameters:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

# Train our model
train_curve, val_curve = train_model(
    net, val_dataloader, train_dataloader, device, epochs, optimizer, criterion)


In [ ]:
plot_learning_curves(train_curve, val_curve, baseline=rand_acc)

Now let's try training our model with a different optimiser and seeing what happens

### Model 2

In [ ]:
reset_seeds(1)
net = Net(D=D) # Set up our model
net.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(net.parameters(), lr=0.001)
train_curve, val_curve = train_model(
    net, val_dataloader, train_dataloader, device, epochs, optimizer, criterion)

In [ ]:
plot_learning_curves(train_curve, val_curve, baseline=rand_acc)

### Model 3

And then let's try a different learning rate

In [ ]:
reset_seeds(1)
net = Net(D=D) # Set up our model
net.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(net.parameters(), lr=0.002)
train_curve, val_curve = train_model(
    net, val_dataloader, train_dataloader, device, epochs, optimizer, criterion)

In [ ]:
plot_learning_curves(train_curve, val_curve, baseline=rand_acc)

### How reliably can we train these models?

We can also try starting with different seeds, and comparing the accuracies we obtain. When making other choices, we want to be sure that we're not just basing them on the variability of the training.

### Seed 2

In [ ]:
reset_seeds(2) # Make things reproducible
net = Net(D=D) # Set up our model
net.to(device) # Send it to the gpu
# Let's choose some hyperparameters:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

# Train our model
train_curve, val_curve = train_model(
    net, val_dataloader, train_dataloader, device, epochs, optimizer, criterion)

In [ ]:
plot_learning_curves(train_curve, val_curve, baseline=rand_acc)

### Seed 3

In [ ]:
reset_seeds(3) # Make things reproducible
net = Net(D=D) # Set up our model
net.to(device) # Send it to the gpu
# Let's choose some hyperparameters:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

# Train our model
train_curve, val_curve = train_model(
    net, val_dataloader, train_dataloader, device, epochs, optimizer, criterion)

In [ ]:
plot_learning_curves(train_curve, val_curve, baseline=rand_acc)

We have to run our experiments multiple times to know whether we are observing noise, or an actual difference!

## Now you're ready to start your projects! 

Now that you've seen the model in action, have a look at how the model is defined in 
`src/define_model.py`, and how it is trained and evaluated in `src/train_and_evaluate.py`.

Here are some things you can experiment with during the projects:
- The choice of optimiser
- The batch size
- The learning rate
- The amount of image compression (see `srcweather_helpers.py`)
- The network definition